# Catalysis AutoResearch — replay (no agent needed)

Step through a **real** run of the harness, one candidate model per cell. Each
cell fits a kinetic model to the data and shows how well it explains the five
observed species — and whether total mass stays at the closed-vessel value of
500 mmol/L. You are the loop: run the cells top to bottom and watch the evidence
(ELBO) climb as a hidden species is introduced.

The models below are exactly the ones a coding agent (Claude Sonnet) proposed,
in order, when told only *"maximise the ELBO."*

In [ ]:
# --- Setup (run once) --------------------------------------------------
try:
    import numpyro  # already installed?
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "numpyro>=0.15", "jax>=0.4", "numpy>=1.26", "matplotlib>=3.8"])

import os, sys
if not os.path.exists("prepare.py"):
    os.system("git clone -q -b extras https://github.com/BoltMaxwell/cecam-demo.git _repo")
    sys.path.insert(0, "_repo"); os.chdir("_repo")

import numpy as np, jax, jax.numpy as jnp
import numpyro, numpyro.distributions as dist
from jax import lax
from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoNormal
import matplotlib.pyplot as plt
import prepare

numpyro.set_platform("cpu")
jax.config.update("jax_transfer_guard", "allow")

TOTAL0 = prepare.TOTAL0
_D = prepare.load(); TIMES, OBS = _D["times"], _D["obs"]
OBS_NORM = jnp.array(OBS / TOTAL0)
T_FINAL, N_STEPS = 180.0, 360
DT = T_FINAL / N_STEPS
MEAS_IDX = np.array([int(round(t / DT)) for t in TIMES])
LOG_RATE_LOC, LOG_RATE_SCALE = jnp.log(0.02), 1.0

_best = {"elbo": None}


def _fit(species, y0, n_rates, rhs):
    obs_cols = jnp.array([species.index(n) for n in prepare.OBSERVED])
    def integrate(k):
        def step(c, i):
            t = i * DT
            a = rhs(c, t, k); b = rhs(c + 0.5*DT*a, t, k)
            d = rhs(c + 0.5*DT*b, t, k); e = rhs(c + DT*d, t, k)
            cn = c + (DT/6.0)*(a + 2*b + 2*d + e); return cn, cn
        _, tr = lax.scan(step, y0, jnp.arange(N_STEPS))
        return jnp.concatenate([y0[None, :], tr], axis=0)
    def model(y):
        log_k = numpyro.sample("log_k", dist.Normal(LOG_RATE_LOC*jnp.ones(n_rates), LOG_RATE_SCALE))
        sigma = numpyro.sample("sigma", dist.HalfNormal(0.1))
        pred = integrate(jnp.exp(log_k))[MEAS_IDX][:, obs_cols]
        numpyro.sample("y", dist.Normal(pred, sigma), obs=y)
    guide = AutoNormal(model)
    svi = SVI(model, guide, numpyro.optim.Adam(5e-3), Trace_ELBO())
    res = svi.run(jax.random.PRNGKey(0), 4000, OBS_NORM, progress_bar=False)
    elbo = float(-Trace_ELBO(num_particles=512).loss(jax.random.PRNGKey(1), res.params, model, guide, OBS_NORM))
    k_hat = jnp.exp(guide.median(res.params)["log_k"])
    traj = np.asarray(integrate(k_hat))
    return elbo, traj


def step_through(title, species, y0, n_rates, rhs):
    """Fit one model, plot the fit + total mass, and report the keep/discard call."""
    elbo, traj = _fit(species, jnp.asarray(y0), n_rates, rhs)
    tr = traj * TOTAL0
    total = tr[np.asarray(MEAS_IDX)].sum(axis=1)
    resid = float(np.max(np.abs(total - TOTAL0)) / TOTAL0)
    passed = resid <= 0.02
    gt = np.arange(tr.shape[0]) * DT
    colors = plt.cm.tab10.colors

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 7), gridspec_kw={"height_ratios": [3, 1]})
    for j, nm in enumerate(species):
        hid = nm not in prepare.OBSERVED
        a1.plot(gt, tr[:, j], color=colors[j % 10], ls="--" if hid else "-",
                label=nm + (" (hidden)" if hid else ""))
    for kk, nm in enumerate(prepare.OBSERVED):
        a1.scatter(TIMES, OBS[:, kk], s=30, zorder=5, edgecolor="k", linewidth=0.4,
                   color=colors[species.index(nm) % 10])
    a1.set_title(f"{title}\nELBO = {elbo:.2f}   |   mass {'PASS' if passed else 'FAIL'}")
    a1.set_ylabel("concentration (mmol/L)"); a1.legend(fontsize=8, ncol=2)
    a2.plot(gt, tr.sum(axis=1), color="k")
    a2.axhline(TOTAL0, color="tab:red", ls=":", label="target 500")
    a2.set_ylabel("total mass"); a2.set_xlabel("time"); a2.legend(fontsize=8)
    fig.tight_layout(); plt.show()

    if _best["elbo"] is None:
        _best["elbo"] = elbo
        print(f"ELBO = {elbo:.2f}   (baseline — the score to beat)")
    else:
        odds = np.exp(elbo - _best["elbo"])
        if odds > 1:
            _best["elbo"] = elbo
            print(f"ELBO = {elbo:.2f}   KEEP  (odds {odds:.2f} > 1 vs best)")
        else:
            print(f"ELBO = {elbo:.2f}   discard  (odds {odds:.2f} < 1 vs best)")
    return elbo


### 0. Baseline — five observed species, no hidden intermediate

The shipped model: NO3 -> NO2, then NO2 splits straight to N2, NH3, N2O. It conserves mass but can't match the observed decline (the observed species sum to < 500), so its evidence is low.

In [ ]:
SPECIES = ["NO3", "NO2", "N2", "NH3", "N2O"]
def rhs(y, t, k):
    NO3, NO2, N2, NH3, N2O = y
    k1, k2, k4, k5 = k
    return jnp.array([-k1*NO3, k1*NO3-(k2+k4+k5)*NO2, k2*NO2, k4*NO2, k5*NO2])
step_through("Baseline: NO3->NO2->{N2, NH3, N2O}, no hidden species",
             SPECIES, [1., 0, 0, 0, 0], 4, rhs)

### 1. Add a hidden **NO** intermediate (series)

The missing mass hints at an unobserved species. Insert hidden NO in a chain: NO2 -> NO -> N2O -> N2, with NH3 still off NO2. The ELBO jumps.

In [ ]:
SPECIES = ["NO3", "NO2", "NO", "N2", "NH3", "N2O"]
def rhs(y, t, k):
    NO3, NO2, NO, N2, NH3, N2O = y
    k1, k2, k4, k5, k6 = k
    dNO3 = -k1*NO3
    dNO2 = k1*NO3 - (k2+k4)*NO2
    dNO  = k2*NO2 - k5*NO
    dN2O = k5*NO - k6*N2O
    dN2  = k6*N2O
    dNH3 = k4*NO2
    return jnp.array([dNO3, dNO2, dNO, dN2, dNH3, dN2O])
step_through("Hidden NO in series: NO2->NO->N2O->N2",
             SPECIES, [1., 0, 0, 0, 0, 0], 5, rhs)

### 2. Let NO branch in **parallel** to N2 and N2O

Instead of a chain, NO splits directly to N2 and N2O (drop the N2O->N2 edge). Fits better still.

In [ ]:
SPECIES = ["NO3", "NO2", "NO", "N2", "NH3", "N2O"]
def rhs(y, t, k):
    NO3, NO2, NO, N2, NH3, N2O = y
    k1, k2, k4, k5, k6 = k
    dNO3 = -k1*NO3
    dNO2 = k1*NO3 - (k2+k4)*NO2
    dNO  = k2*NO2 - (k5+k6)*NO
    dN2  = k5*NO
    dNH3 = k4*NO2
    dN2O = k6*NO
    return jnp.array([dNO3, dNO2, dNO, dN2, dNH3, dN2O])
step_through("Hidden NO branches in parallel to N2 and N2O",
             SPECIES, [1., 0, 0, 0, 0, 0], 5, rhs)

### 3. Hybrid — also drain N2O -> N2 (a superset)

Add the extra N2O->N2 edge back on top of the parallel branching. More parameters, but does the evidence justify them?

In [ ]:
SPECIES = ["NO3", "NO2", "NO", "N2", "NH3", "N2O"]
def rhs(y, t, k):
    NO3, NO2, NO, N2, NH3, N2O = y
    k1, k2, k4, k5, k6, k7 = k
    dNO3 = -k1*NO3
    dNO2 = k1*NO3 - (k2+k4)*NO2
    dNO  = k2*NO2 - (k5+k6)*NO
    dN2O = k6*NO - k7*N2O
    dN2  = k5*NO + k7*N2O
    dNH3 = k4*NO2
    return jnp.array([dNO3, dNO2, dNO, dN2, dNH3, dN2O])
step_through("Hybrid: parallel NO branching + N2O->N2",
             SPECIES, [1., 0, 0, 0, 0, 0], 6, rhs)

### 4. Route **NH3 off NO** too — three-way branch (best)

Source NH3 from hidden NO instead of NO2, so NO is a clean three-way branch point (N2, NH3, N2O). This wins the evidence and becomes the best model.

In [ ]:
SPECIES = ["NO3", "NO2", "NO", "N2", "NH3", "N2O"]
def rhs(y, t, k):
    NO3, NO2, NO, N2, NH3, N2O = y
    k1, k2, k4, k5, k6 = k
    dNO3 = -k1*NO3
    dNO2 = k1*NO3 - k2*NO2
    dNO  = k2*NO2 - (k4+k5+k6)*NO
    dN2  = k5*NO
    dNH3 = k4*NO
    dN2O = k6*NO
    return jnp.array([dNO3, dNO2, dNO, dN2, dNH3, dN2O])
step_through("Hidden NO branches three ways: N2, NH3, N2O  (BEST)",
             SPECIES, [1., 0, 0, 0, 0, 0], 5, rhs)

### 5. Add a direct NO2 -> N2 shortcut (rejected)

Try letting some N2 form directly from NO2, bypassing hidden NO. It fits a touch differently but doesn't beat the best — the evidence rejects it.

In [ ]:
SPECIES = ["NO3", "NO2", "NO", "N2", "NH3", "N2O"]
def rhs(y, t, k):
    NO3, NO2, NO, N2, NH3, N2O = y
    k1, k2, k3, k4, k5, k6 = k
    dNO3 = -k1*NO3
    dNO2 = k1*NO3 - (k2+k3)*NO2
    dNO  = k2*NO2 - (k4+k5+k6)*NO
    dN2  = k3*NO2 + k5*NO
    dNH3 = k4*NO
    dN2O = k6*NO
    return jnp.array([dNO3, dNO2, dNO, dN2, dNH3, dN2O])
step_through("Add direct NO2->N2 shortcut (bypasses NO)",
             SPECIES, [1., 0, 0, 0, 0, 0], 6, rhs)

### 6. Add a *second* hidden species NOH (rejected)

Insert another unobserved species (NOH) in series before NO. A second hidden species does not improve the evidence — the data support exactly one.

In [ ]:
SPECIES = ["NO3", "NO2", "NOH", "NO", "N2", "NH3", "N2O"]
def rhs(y, t, k):
    NO3, NO2, NOH, NO, N2, NH3, N2O = y
    k1, k2, k4, k5, k6, k7 = k
    dNO3 = -k1*NO3
    dNO2 = k1*NO3 - k2*NO2
    dNOH = k2*NO2 - k7*NOH
    dNO  = k7*NOH - (k4+k5+k6)*NO
    dN2  = k5*NO
    dNH3 = k4*NO
    dN2O = k6*NO
    return jnp.array([dNO3, dNO2, dNOH, dNO, dN2, dNH3, dN2O])
step_through("Second hidden species NOH in series before NO",
             SPECIES, [1., 0, 0, 0, 0, 0, 0], 6, rhs)

## Takeaways

- Told only *"maximise the ELBO,"* the search **discovered a hidden species (NO)**
  it was never told about — the only honest way to explain the missing mass —
  and the evidence climbed from ~44 to ~67.
- Mass stayed conserved (PASS) at every step: the hidden species *stores* the
  missing mass rather than destroying it.
- The **rejections matter**: extra edges and a second hidden species fit slightly
  better but didn't beat the evidence penalty, so the ratchet discarded them.
  That is the difference between fitting and *science*.

A different agent or model may reach a different valid answer — the point is that
the harness makes the evidence, not the metric, decide.